In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation
from xgboost import XGBClassifier


In [2]:
train_raw = pd.read_csv('home-credit-default-risk/application_train.csv', low_memory=False)
test_raw = pd.read_csv('home-credit-default-risk/application_test.csv', low_memory=False)
imputed_full = pd.read_csv('imputed_data.csv', index_col=0)

for frame in (train_raw, test_raw):
    frame['SK_ID_CURR'] = pd.to_numeric(frame['SK_ID_CURR'], errors='coerce')
    frame.dropna(subset=['SK_ID_CURR'], inplace=True)
    frame['SK_ID_CURR'] = frame['SK_ID_CURR'].astype('int64')

imputed_full['SK_ID_CURR'] = pd.to_numeric(imputed_full['SK_ID_CURR'], errors='coerce').astype('int64')
imputed_indexed = imputed_full.set_index('SK_ID_CURR')

y = train_raw['TARGET'].astype(int)
train_ids = train_raw['SK_ID_CURR'].to_numpy()
test_ids = test_raw['SK_ID_CURR'].to_numpy()

missing_train = set(train_ids) - set(imputed_indexed.index)
missing_test = set(test_ids) - set(imputed_indexed.index)
if missing_train:
    raise ValueError(f'imputed_data 缺少 {len(missing_train)} 個訓練 SK_ID_CURR。')
if missing_test:
    raise ValueError(f'imputed_data 缺少 {len(missing_test)} 個測試 SK_ID_CURR。')

imputed_train = imputed_indexed.reindex(train_ids).reset_index()
imputed_test = imputed_indexed.reindex(test_ids).reset_index()

train_orig = train_raw.drop(columns=['TARGET']).copy()
test_orig = test_raw.copy()

train_merged = train_orig.merge(
    imputed_train,
    on='SK_ID_CURR',
    how='left',
    suffixes=('_orig', '_imputed')
)
test_merged = test_orig.merge(
    imputed_test,
    on='SK_ID_CURR',
    how='left',
    suffixes=('_orig', '_imputed')
)

def prepare_features(train_df, test_df, prefix):
    train_noid = train_df.drop(columns=['SK_ID_CURR']).copy()
    test_noid = test_df.drop(columns=['SK_ID_CURR']).copy()
    combined = pd.concat([train_noid, test_noid], axis=0, ignore_index=True)
    categorical_cols = combined.select_dtypes(include=['object']).columns.tolist()
    if categorical_cols:
        print(f'[{prefix}] One-hot encoding {len(categorical_cols)} object columns')
        combined = pd.get_dummies(combined, columns=categorical_cols, dummy_na=False)
    else:
        print(f'[{prefix}] No object columns detected.')
    combined = combined.astype('float32')
    train_rows = len(train_noid)
    train_features = combined.iloc[:train_rows].copy()
    test_features = combined.iloc[train_rows:].copy()
    feature_names = [f'{prefix}_f_{i}' for i in range(train_features.shape[1])]
    train_features.columns = feature_names
    test_features.columns = feature_names
    print(f'[{prefix}] Feature count: {len(feature_names)}')
    return train_features, test_features, feature_names

feature_sets = {}
orig_train_features, orig_test_features, orig_cols = prepare_features(train_orig, test_orig, 'orig')
feature_sets['orig'] = {
    'train': orig_train_features,
    'test': orig_test_features,
    'feature_cols': orig_cols,
    'label': 'Original only',
    'submission_path': 'lgbm_orig_submission.csv'
}

merged_train_features, merged_test_features, merged_cols = prepare_features(train_merged, test_merged, 'merged')
feature_sets['merged'] = {
    'train': merged_train_features,
    'test': merged_test_features,
    'feature_cols': merged_cols,
    'label': 'Original + imputed',
    'submission_path': 'lgbm_merged_submission.csv'
}

print('Train rows:', len(y))
print('Test rows:', len(test_ids))


[orig] One-hot encoding 16 object columns
[orig] Feature count: 244
[merged] One-hot encoding 16 object columns
[merged] Feature count: 373
Train rows: 307511
Test rows: 48744


In [46]:
# LightGBM modelling for both feature sets
feature_order = ['orig', 'merged']
lgbm_predictions = {}
lgbm_metrics = {}

for feature_key in feature_order:
    if feature_key not in feature_sets:
        continue
    cfg = feature_sets[feature_key]
    print(f"=== LightGBM on {cfg['label']} ({feature_key}) ===")

    X_current = cfg['train']
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_current, y, test_size=0.2, random_state=42, stratify=y
    )

    X_train_np = X_train.to_numpy(dtype='float32')
    X_valid_np = X_valid.to_numpy(dtype='float32')
    test_matrix = cfg['test'].to_numpy(dtype='float32')

    lgb_params = {
        'n_estimators': 2000,
        'learning_rate': 0.03,
        'max_depth': -1,
        'num_leaves': 64,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_lambda': 1.0,
        'random_state': 42,
        'n_jobs': -1,
        'objective': 'binary',
        'metric': 'auc',
    }

    lgb_model = LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_train_np, y_train,
        eval_set=[(X_valid_np, y_valid)],
        feature_name='auto',
        callbacks=[early_stopping(stopping_rounds=100), log_evaluation(period=100)],
    )

    valid_probs = lgb_model.predict_proba(X_valid_np)[:, 1]
    valid_auc = roc_auc_score(y_valid, valid_probs)
    best_iter = lgb_model.best_iteration_ or lgb_params['n_estimators']
    print(f"LightGBM validation ROC AUC: {valid_auc:.4f}")
    print(f"Best iteration: {best_iter}")

    final_lgb = LGBMClassifier(**{**lgb_params, 'n_estimators': best_iter})
    X_full = X_current.to_numpy(dtype='float32')
    final_lgb.fit(X_full, y, feature_name='auto')
    test_probs = final_lgb.predict_proba(test_matrix)[:, 1]
    submission_df = pd.DataFrame({
        'SK_ID_CURR': test_ids,
        'TARGET': test_probs,
    })

    lgbm_predictions[feature_key] = submission_df
    lgbm_metrics[feature_key] = {'valid_auc': valid_auc, 'best_iter': best_iter}

# default export uses merged features if available, otherwise fall back to last run
if 'merged' in lgbm_predictions:
    model_submission = lgbm_predictions['merged']
    print('model_submission set to LightGBM merged predictions')
elif lgbm_predictions:
    last_key = feature_order[-1] if feature_order[-1] in lgbm_predictions else next(iter(lgbm_predictions))
    model_submission = lgbm_predictions[last_key]
    print(f'model_submission set to LightGBM predictions of {last_key}')
else:
    raise RuntimeError('LightGBM produced no predictions. Check feature_sets.')

print('LightGBM runs summary:', lgbm_metrics)


=== LightGBM on Original only (orig) ===
[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.060936 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11478
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 233
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.753629
[200]	valid_0's auc: 0.76037
[300]	valid_0's auc: 0.761566
[400]	valid_0's auc: 0.761668
[500]	valid_0's auc: 0.762102
[600]	valid_0's auc: 0.762346
[700]	valid_0's auc: 0.762505
[800]	valid_0's auc: 0.762611
Early stopping, best iteration is:
[760]	valid_0's auc: 0.762669
LightGBM validation ROC AUC: 0.7627
Best iteration: 760
[LightGBM] [Info] Number of positive: 2

In [47]:
# XGBoost modelling for both feature sets (overwrites model_submission)
feature_order = ['orig', 'merged']
xgb_predictions = {}
xgb_metrics = {}

xgb_params = {
    'n_estimators': 600,
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'reg_lambda': 1.0,
    'n_jobs': -1,
    'random_state': 42,
}

for feature_key in feature_order:
    if feature_key not in feature_sets:
        continue
    cfg = feature_sets[feature_key]
    print(f"=== XGBoost on {cfg['label']} ({feature_key}) ===")

    X_current = cfg['train']
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_current, y, test_size=0.2, random_state=42, stratify=y
    )

    X_train_np = X_train.to_numpy(dtype='float32')
    X_valid_np = X_valid.to_numpy(dtype='float32')
    test_matrix = cfg['test'].to_numpy(dtype='float32')

    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train_np, y_train,
        eval_set=[(X_valid_np, y_valid)],
        verbose=100,
    )

    valid_probs = xgb_model.predict_proba(X_valid_np)[:, 1]
    valid_auc = roc_auc_score(y_valid, valid_probs)
    best_iter = getattr(xgb_model, 'best_iteration', None)
    best_n_estimators = (best_iter + 1) if best_iter is not None else xgb_params['n_estimators']
    print(f"XGBoost validation ROC AUC: {valid_auc:.4f}")
    print(f"Best iteration: {best_n_estimators}")

    final_xgb = XGBClassifier(**{**xgb_params, 'n_estimators': best_n_estimators})
    X_full = X_current.to_numpy(dtype='float32')
    final_xgb.fit(X_full, y)
    test_probs = final_xgb.predict_proba(test_matrix)[:, 1]
    submission_df = pd.DataFrame({
        'SK_ID_CURR': test_ids,
        'TARGET': test_probs,
    })

    xgb_predictions[feature_key] = submission_df
    xgb_metrics[feature_key] = {'valid_auc': valid_auc, 'best_iter': best_n_estimators}

# default export uses merged predictions if available
if 'merged' in xgb_predictions:
    model_submission = xgb_predictions['merged']
    print('model_submission set to XGBoost merged predictions')
elif xgb_predictions:
    last_key = feature_order[-1] if feature_order[-1] in xgb_predictions else next(iter(xgb_predictions))
    model_submission = xgb_predictions[last_key]
    print(f'model_submission set to XGBoost predictions of {last_key}')
else:
    raise RuntimeError('XGBoost produced no predictions. Check feature_sets.')

print('XGBoost runs summary:', xgb_metrics)


=== XGBoost on Original only (orig) ===
[0]	validation_0-auc:0.71548
[100]	validation_0-auc:0.75598
[200]	validation_0-auc:0.76080
[300]	validation_0-auc:0.76230
[400]	validation_0-auc:0.76279
[500]	validation_0-auc:0.76252
[599]	validation_0-auc:0.76242
XGBoost validation ROC AUC: 0.7624
Best iteration: 600
=== XGBoost on Original + imputed (merged) ===
[0]	validation_0-auc:0.71533
[100]	validation_0-auc:0.75811
[200]	validation_0-auc:0.76357
[300]	validation_0-auc:0.76473
[400]	validation_0-auc:0.76567
[500]	validation_0-auc:0.76553
[599]	validation_0-auc:0.76505
XGBoost validation ROC AUC: 0.7650
Best iteration: 600
model_submission set to XGBoost merged predictions
XGBoost runs summary: {'orig': {'valid_auc': 0.7624238661397049, 'best_iter': 600}, 'merged': {'valid_auc': 0.7650476181621131, 'best_iter': 600}}


In [ ]:
# Save whichever model_submission was last created
output_path = 'model_submission.csv'
if 'model_submission' not in globals():
    raise RuntimeError('model_submission is not defined. Run one of the modelling cells first.')
model_submission.to_csv(output_path, index=False)
print(f'Saved current model_submission to {output_path} with shape {model_submission.shape}')
